********************************************************************************

PORTFOLIO INVESTOR CODE - MAIN

********************************************************************************

In [ ]:
# Code to portfolio and assets management in an Google Sheets file
# This is an integrated system with Google Sheets and python code working togheter
#
# ENVIRONMENT
# This code was developed to VS CODE and GOOGLE COLAB environments
# According the environment the code needs some changes to access data, so the blocks and lines to be commented and/or uncommented are requesred in code
# Some SET UP comments and tips maybe are in the end of the code
#
# GOOGLE SHEETS FILE
# portfolio  - Google Sheets file name with all data on sheets inside
#
# GOOGLE SHEETS in file:
#
# PANEL SHEET
# panel sheet - It is strongly recomended to create a sheet panel to consolidate quotes from Google Finance and the values from historical series as well as you can bring some key fundamentalists infos
# In panel is where the information concentrates (inputs and outputs) in order to a unified vision and information to strategies such as quotes, min, max, DCF and PE valuations, etc.
# Create in panel your own methods to calculate future returns and link with returns in portfolioIn and portfoliosimIn also, it helps in simulations calculations
# The sheet panel can be customized according user desire
#
# OTHER SHEETS
# parameters sheet - general parameters inputs to used in portfolio and portfolio simulation calculations
# - RiskFree - in decimal float number that representes the percentage, ex. 7.54 means 7.54%
# - Period - period to series reading (1y, 2y, ...)
# - TargetSharpe - percentage of maximum sharpe, in float number that representes the percentage, ex. 85 means 85%
# - TargetSharpeSim - percentage of maximum sharpe, in float number, to portfolio simulation
# tickersIn sheet - with all tickers in lines organized by coluns: StockBr, ReitBr, StockUs, EtfUs,	ReitUs. Dont needed ".SA" in BR tickers, the code generates when accessing YF.
# portfolioIn sheet- protfolio inputs values
# portfoliosimIn sheet - protfolio simulation inputs values
# stockbrHist sheet - BR stocks output factors from historical series in BRL
# reitbrHist sheet - BR REITs (FIIs) output actors from historical series in BRL
# stockusHist sheet - US stocks factors output from historical series in USD
# etfusHist sheet - US ETFs factors output from historical series in USD
# reitusHist sheet - US REITs factors output from historical series in USD
# portfolio sheet - portfolio assets key numbers output in BRL (US assets are converted to BRL)
# portfoliosim sheet - portfolio simulation assets key numbers output in BRL (US assets are converted to BRL)
# stockbrFund sheet - stockbr assets fundamentalist data output
# reitbrFund sheet - reitbr assets fundamentalists data output
# stockusFund sheet - stockus assets fundamentalists data output
# reitusFund sheet - reitus assets fundamentalists data output
#
# EXCEL EXPORT
# quotes sheet - As optional is suggested to create a sheet quotes as a mirror output of informations you desire to export to excel
# in quotes you can combine outputs from panel or other sheets readings such as quotations, risk free, CAPM SML, DYs, etc.
#
# DATA SOURCES:
# Google Finance - To real time quotations in quotes sheet
# Yahoo Finance - To assets series, BR assets needs '.SA" added to the ticker. Yahoo Finance provide US assets fundamentalists data, but not in case o BR assets
# Funds Explorer - Te code scrape data from https://www.fundsexplorer.com.br/ranking .
# In case of site changes and scraping break, as optional copy and paste only values data in an excel spreadsheet FundsExplorer.xlsx and save in Google Drive.
# The code will read this file instead of scraping the site. The code uses reitbrDataFund function to scrape data from Funds Explorer site and reitbrData function to read data from FundsExplorer.xlsx file.
# Funds Explorer file should be in a Google Drive directory which the the path is on code, adjust the path on code if needed and give file access to code
#
# IMPORTANT
# Create and save .py specific package files in the correct path to the this main code
# Make activities related to data source before run the code
# The code uses XFIX11 as a proxy of IFIX
# Verifiy data of source files files in case of code running problems
# During the code running it is necessary to allow access to Google, necessary code access Google Drive where files are located
# Check key moments in code, use display and prints commented in code if needed
# - libraries importations
# - Google Drive and Google Sheets accesses
# - portfolio and portfolio sim dataframe creations
# - parameters and assets lists generation
# - historical data series readings
# - portfolio dataframe assembling
# - maximum and target sharpe numbers portfolio adding
# - assets fundamentalists infos readings and writing on sheets
# You can export the sheets to excel, essentially the sheets portfolio and quotes to manage your specific assets and strategies
# Fundamentalists infos do not change in short time, so FundReading = "Yes" to the first running, after suggested to change to "No", it takes processing time
# Import portfolio simulation to excel and simulate with different assets, weights and limits, avoid use portofolio sheet to simulation
# Mandatory the firsts registers to be IBOV and USDBRL to both, portfolio and portfolio simulation
# Portofolio and Portfolio Simulation limits are only Target Sharpe and its weights restriction condition, real porfolio can have different weights
# It is suggested to limit tickers in not to much to, remeber IBOV and USDBRL the 2 initial ones

********************************************************************************
LIBRARIES IMPORTS
********************************************************************************

In [ ]:
# standard packages imports
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import yfinance as yf
import datetime as dt

In [ ]:
# Google access packages
import io
import sys
import os
import gspread

In [ ]:
# VS CODE ONLY to Google Drive access, comment the block in case not using VS Code
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaIoBaseDownload
# from google.oauth2.service_account import Credentials

In [ ]:
# COLAB CODE ONLY to Google Drive access, comment this block in case not using Colab
from google.colab import auth
from google.auth import default
from google.colab import drive

In [ ]:
# COLAB CODE ONLY to Google Drive access, comment the block in case not using Colab
# drive mount and path to modules
drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/Colab Notebooks/portfolio')
os.chdir('/content/drive/MyDrive/Colab Notebooks/portfolio')# COLAB CODE ONLY to Google Drive access
print("Diretório atual:", os.getcwd())

In [ ]:
# specific (modules) packages import functions
# BR and US data series reading and US assets info reading
from modules.yahoofinance import yfSeries, yfStock, yfEtf
# BR reits (FIIs) series and info readings
from modules.reitbr import reitbrData, reitbrDataFund
# converts USD columns to BRL
from modules.usdbrl import usdToBrl
# calculate assets factors - min, max return, risk and beta
from modules.assetsfactors import assetsFactors
# calculate portfolio performance
from modules.portfolioperformance import portfolioPerformance, maxSharpeWeights, targetSharpeWeights

********************************************************************************
GOOGLE SHEETS READINGS
********************************************************************************

In [ ]:
# VS CODE ONLY to Google Drive access, comment the block in case not using VS Code
# scopes and paths
# scopes = ['https://www.googleapis.com/auth/drive','https://www.googleapis.com/auth/spreadsheets']
# credentials json path
# jasonpath = '/home/maber/keys/googlekey.json'
# Autenticação centralizada para os dois serviços
# credentials = Credentials.from_service_account_file(jasonpath, scopes=scopes)

In [ ]:
# VS CODE ONLY to Google Drive access, comment the block in case not using VS Code
# Google Sheets and Drive connection
# gc = gspread.authorize(credentials)
# sh = gc.open('portfolio')

In [ ]:
# COLAB CODE ONLY to Google Drive access, comment the block in case not using Colab
# Google Sheets connection
auth.authenticate_user()
credentials, _ = default()

In [ ]:
# COLAB CODE ONLY to Google Drive access, comment the block in case not using Colab
# Google Sheets and Drive connection
gc = gspread.authorize(credentials)
sh = gc.open('portfolio')

In [ ]:
# generic reading function
def read_sheet(tab):
    ws = sh.worksheet(tab)
    df = pd.DataFrame(ws.get_all_records())
    return df

In [ ]:
# generic parameters reading
parameters = read_sheet('parametersIn')

# extracting float values
risk_free = float(parameters.loc[parameters['Parameter'] == 'RiskFree', 'Value'].values[0]) / 100
target_sharpe = float(parameters.loc[parameters['Parameter'] == 'TargetSharpe', 'Value'].values[0]) / 100
target_sharpesim = float(parameters.loc[parameters['Parameter'] == 'TargetSharpeSim', 'Value'].values[0]) / 100

# extracting string
period = str(parameters.loc[parameters['Parameter'] == 'Period', 'Value'].values[0])
fund_reading = str(parameters.loc[parameters['Parameter'] == 'FundReading', 'Value'].values[0])

print(f'risk_free =', risk_free,'  period=' ,period, '  target_sharpe =', target_sharpe, '  info_reading =', fund_reading)

In [ ]:
# portfolio input reading
portfolio = read_sheet('portfolioIn')

# adjustig dataframe columns data
portfolio = portfolio[portfolio['ticker'] != '']
portfolio['return'] = portfolio['return'].astype(float) / 10
portfolio['weight'] = portfolio['weight'].astype(float) / 1000
portfolio['limit'] = portfolio['limit'].astype(float) / 10

display(portfolio)

# if currency = 'BRL', add '.SA' to ticker name
# portlist_df is a temporary dataframe only to generate adequate list
portlist_df = portfolio[['currency','ticker']].copy()
portlist_df['ticker'] = portlist_df.apply(
    lambda row: f"{row['ticker']}.SA" if row['currency'] == 'BRL' else row['ticker'],
    axis=1
)
# display(portlist_df)

#
port_ticker = portlist_df['ticker'].dropna().tolist()
port_ticker = [t for t in port_ticker if str(t).strip() != ""]

# '.SA' adding and replace IBOV to '^BVSP^ and USBBRL to USDBRL=X
# port_ticker = [nome + '.SA' for nome in port_ticker]
port_ticker = ['^BVSP' if nome == 'IBOV.SA' else nome for nome in port_ticker]
port_ticker = ['USDBRL=X' if nome == 'USDBRL.SA' else nome for nome in port_ticker]
print(f'port_ticker =', port_ticker)

# expected return list
port_expret = portfolio['return'].tolist()
print(f'port_expret =', port_expret)

# weights list
port_weight = portfolio['weight'].tolist()
print(f'port_weight =', port_weight)

# limits list, maximum asset weight acceptable in portfolio
port_limits = portfolio['limit'].tolist()
print(f'port_limits = ', port_limits)

In [ ]:
# portfolio simulation input reading
portfoliosim = read_sheet('portfoliosimIn')

# adjustig dataframe columns data
portfoliosim = portfoliosim[portfoliosim['ticker'] != '']
portfoliosim['return'] = portfoliosim['return'].astype(float) / 10
portfoliosim['weight'] = portfoliosim['weight'].astype(float) / 1000
portfoliosim['limit'] = portfoliosim['limit'].astype(float) / 10

display(portfoliosim)

# if currency = 'BRL', add '.SA' to ticker name
# portlist_df is a temporary dataframe only to generate adequate list
portsimlist_df = portfoliosim[['currency','ticker']].copy()
portsimlist_df['ticker'] = portsimlist_df.apply(
    lambda row: f"{row['ticker']}.SA" if row['currency'] == 'BRL' else row['ticker'],
    axis=1
)
# display(portlist_df)

#
portsim_ticker = portsimlist_df['ticker'].dropna().tolist()
portsim_ticker = [t for t in port_ticker if str(t).strip() != ""]

# '.SA' adding and replace IBOV to '^BVSP^ and USBBRL to USDBRL=X
# port_ticker = [nome + '.SA' for nome in port_ticker]
portsim_ticker = ['^BVSP' if nome == 'IBOV.SA' else nome for nome in portsim_ticker]
portsim_ticker = ['USDBRL=X' if nome == 'USDBRL.SA' else nome for nome in portsim_ticker]
print(f'portsim_ticker =', portsim_ticker)

# expected return list
portsim_expret = portfoliosim['return'].tolist()
print(f'portsim_expret =', portsim_expret)

# weights list
portsim_weight = portfoliosim['weight'].tolist()
print(f'portsim_weight =', portsim_weight)

# limits list, maximum asset weight acceptable in portfolio simulation
portsim_limits = portfoliosim['limit'].tolist()
print(f'portsim_limits = ', portsim_limits)

In [ ]:
# tickers reading
ticker_df = read_sheet("tickersIn")

# remove linhas completamente vazias
ticker_df = ticker_df.dropna(how="all")

def get_tickers(col):
    return ticker_df[col].dropna().tolist()

# lists creation
stockbr_ticker = get_tickers('StockBr')
reitbr_ticker  = get_tickers('ReitBr')
stockus_ticker = get_tickers('StockUs')
etfus_ticker   = get_tickers('EtfUs')
reitus_ticker  = get_tickers('ReitUs')

# clean lists empty spaces
stockbr_ticker = [item for item in stockbr_ticker if item is not None and item != '']
reitbr_ticker = [item for item in reitbr_ticker if item is not None and item != '']
stockus_ticker = [item for item in stockus_ticker if item is not None and item != '']
etfus_ticker   = [item for item in etfus_ticker if item is not None and item != '']
reitus_ticker  = [item for item in reitus_ticker if item is not None and item != '']

# conformation lists to historical Yahoo Finance series download preparation
# '.SA' adding and replace IBOV to '^BVSP^ to StockBr list
stockbr_ticker = [nome + '.SA' for nome in stockbr_ticker]
stockbr_ticker = ['^BVSP' if nome == 'IBOV.SA' else nome for nome in stockbr_ticker]
# '.SA' adding and exclude IFIX from ReitBr list
reitbr_ticker = [nome + '.SA' for nome in reitbr_ticker]
# reitbr_ticker = [nome for nome in reitbr_ticker if nome != 'IFIX.SA']
# replace SP500 by ^GSPC and USDBRL by USDBRL=X on lists StockUS and EtfUS
stockus_ticker = ['^GSPC' if nome == 'SP500' else nome for nome in stockus_ticker]
stockus_ticker = ['USDBRL=X' if nome == 'USDBRL' else nome for nome in stockus_ticker]
etfus_ticker = ['^GSPC' if nome == 'SP500' else nome for nome in etfus_ticker]
etfus_ticker = ['USDBRL=X' if nome == 'USDBRL' else nome for nome in etfus_ticker]
# replace USDBRL by USDBRL=X on list ReistUS
reitus_ticker = ['USDBRL=X' if nome == 'USDBRL' else nome for nome in reitus_ticker]

# print checking
print(f'stockbr_ticker =', stockbr_ticker)
print(f'reitbr_ticker =', reitbr_ticker)
print(f'stockus_ticker =', stockus_ticker)
print(f'etfus_ticker =', etfus_ticker)
print(f'reitus_ticker =', reitus_ticker)

********************************************************************************
DATA SERIES READINGS
********************************************************************************

In [ ]:
# READING FROM DATA SERIES SOURCES

# series reading from Yahoo Finance
stockbr_series = yfSeries(stockbr_ticker, period=period)
reitbr_series = yfSeries(reitbr_ticker, period=period)
stockus_series = yfSeries(stockus_ticker, period=period)
etfus_series = yfSeries(etfus_ticker, period=period)
reitus_series = yfSeries(reitus_ticker, period=period)
port_series = yfSeries(port_ticker, period=period)
portsim_series = yfSeries(port_ticker, period=period)
# display(stockbr_series)

In [ ]:
# timeline harmonization, same period of time to all series
common_idx = (
    stockbr_series.index
    .intersection(reitbr_series.index)
    .intersection(stockus_series.index)
    .intersection(etfus_series.index)
    .intersection(reitus_series.index)
    .intersection(port_series.index)
    .intersection(portsim_series.index)
)

# Reindex and filling (without warnings)
stockbr_series = stockbr_series.reindex(common_idx).ffill()
reitbr_series = reitbr_series.reindex(common_idx).ffill()
stockus_series = stockus_series.reindex(common_idx).ffill()
etfus_series = etfus_series.reindex(common_idx).ffill()
reitus_series = reitus_series.reindex(common_idx).ffill()
port_series = port_series.reindex(common_idx).ffill()
portsim_series = port_series.reindex(common_idx).ffill()
# display(stockbr_series)

In [ ]:
# converts portfolio series USD columns to BRL
# use from invest.usdbrl import usdToBrl
port_series = usdToBrl(port_series, stockus_series)
port_series = usdToBrl(port_series, etfus_series)
port_series = usdToBrl(port_series, reitus_series)
portsim_series = usdToBrl(portsim_series, stockus_series)
portsim_series = usdToBrl(portsim_series, etfus_series)
portsim_series = usdToBrl(portsim_series, reitus_series)
# display(port_series)

In [ ]:
# calculate assets factors
# use from invest.assetsfactors import assetsFactors
stockbr_factors = assetsFactors(stockbr_series)
reitbr_factors = assetsFactors(reitbr_series)
stockus_factors = assetsFactors(stockus_series)
etfus_factors = assetsFactors(etfus_series)
reitus_factors = assetsFactors(reitus_series)
port_factors = assetsFactors(port_series)
portsim_factors = assetsFactors(port_series)
# display(port_factors)

********************************************************************************
PORTFOLIO DATAFRAME ASSEMBLING AND PERFORMANCE CALCULATION
********************************************************************************

In [ ]:
# portfolio dataframe assembling
portfolio = pd.merge(portfolio, port_factors, on='ticker', how='inner')
# display(portfolio)
portfoliosim = pd.merge(portfoliosim, portsim_factors, on='ticker', how='inner')
# display(portfoliosim)

In [ ]:
# PORTFOLIO PERFORMANCE
# uses from invest.portfolioperformance import portfolioPerformance, maxSharpeWeights, targetSharpeWeights

In [ ]:
riskfree = risk_free / 100 # Risk Free is indicated in percentage

In [ ]:
# daily variation
port_variation = port_series.pct_change()
portsim_variation = portsim_series.pct_change()
# portfolio covariance calculation
port_covariance = port_variation.cov()*252
portsim_covariance = portsim_variation.cov()*252
# convert covariance dataframe in numpy matrix
cov = port_covariance.values
covsim = portsim_covariance.values
# display(cov)

In [ ]:
# weights array
weights = [values / 100 for values in port_weight]
weightssim = [values / 100 for values in portsim_weight]
# extracting historical real returns
mu = portfolio['returnHist'].values / 100
musim = portfoliosim['returnHist'].values / 100

In [ ]:
# porfolio Historical Return, Risk and Sharpe calculations
portfoliorettotal, portfoliorisktotal, portfoliosharpe = portfolioPerformance(weights, mu, cov, riskfree)
portfoliosimrettotal, portfoliosimrisktotal, portfoliosimsharpe = portfolioPerformance(weightssim, musim, covsim, riskfree)
# print (portfoliorettotal, portfoliorisktotal, portfoliosharpe)

In [ ]:
# portfolio adding performance columns in first register(in the same line of IBOV value index)
# other register lines being filled with zero and adequate rounds to values
# Total Historical Return adding
portfolio['returnHistTot'] = [portfoliorettotal * 100] + [0] * (len(portfolio) - 1)
portfoliosim['returnHistTot'] = [portfoliosimrettotal * 100] + [0] * (len(portfoliosim) - 1)
portfolio['returnHistTot'] = portfolio['returnHistTot'].round(1)
portfoliosim['returnHistTot'] = portfoliosim['returnHistTot'].round(1)
# Total Historical Risk adding
portfolio['riskTot'] = [portfoliorisktotal * 100] + [0] * (len(portfolio) - 1)
portfoliosim['riskTot'] = [portfoliosimrisktotal * 100] + [0] * (len(portfoliosim) - 1)
portfolio['riskTot'] = portfolio['riskTot'].round(1)
portfoliosim['riskTot'] = portfoliosim['riskTot'].round(1)
# Total Beta adding
portfolio['betaTot'] = [(portfolio['weight'] / 100 * portfolio['beta']).sum()] + [0] * (len(portfolio) - 1)
portfoliosim['betaTot'] = [(portfoliosim['weight'] / 100 * portfoliosim['beta']).sum()] + [0] * (len(portfoliosim) - 1)
portfolio['betaTot'] = portfolio['betaTot'].round(3)
portfoliosim['betaTot'] = portfoliosim['betaTot'].round(3)
# Total Historical Sharpe adding
portfolio['sharpeHist'] = [portfoliosharpe] + [0] * (len(portfolio) - 1)
portfoliosim['sharpeHist'] = [portfoliosimsharpe] + [0] * (len(portfoliosim) - 1)
portfolio['sharpeHist'] = portfolio['sharpeHist'].round(3)
portfoliosim['sharpeHist'] = portfoliosim['sharpeHist'].round(3)
# display (portfolio)

In [ ]:
# drop Min and Max
portfolio = portfolio.drop(columns=['min', 'max'])
portfoliosim = portfoliosim.drop(columns=['min', 'max'])
# displace Expected Return to rigth, just to organization
cols = list(portfolio.columns)
colssim = list(portfoliosim.columns)
i = cols.index('return')
isim = colssim.index('return')
n = 4
nsim = n
new_order = cols[:i] + cols[i+1 : i+1+n] + [cols[i]] + cols[i+1+n:]
new_ordersim = colssim[:isim] + colssim[isim+1 : isim+1+nsim] + [colssim[isim]] + colssim[isim+1+nsim:]
portfolio = portfolio[new_order]
portfoliosim = portfoliosim[new_ordersim]
# display (portfolio)

In [ ]:
# extracting expected returns
mu = portfolio['return'].values / 100
musim = portfoliosim['return'].values / 100

In [ ]:
# porfolio Expected Return, Expected Risk and Expected Sharpe calculations
portfolioretexptotal, portfolioriskexptotal, portfoliosharpeexp = portfolioPerformance(weights, mu, cov, riskfree)
portfoliosimretexptotal, portfoliosimiskexptotal, portfoliosimsharpeexp = portfolioPerformance(weightssim, musim, covsim, riskfree)
# print (portfolioretexptotal, portfolioriskexptotal, portfoliosharpeexp)

In [ ]:
# portfolio adding columns with Total Expected Return and Total Expected Sharpe values in first registration, the same line of IBOV value index
# other register lines being filled with zero and adequate rounds to values
# Total Expected Return
portfolio['returnTot'] = [portfolioretexptotal * 100] + [0] * (len(portfolio) - 1)
portfoliosim['returnTot'] = [portfoliosimretexptotal * 100] + [0] * (len(portfoliosim) - 1)
portfolio['returnTot'] = portfolio['returnTot'].round(1)
portfoliosim['returnTot'] = portfoliosim['returnTot'].round(1)
# total Sharpe
portfolio['sharpeTot'] = [portfoliosharpeexp] + [0] * (len(portfolio) - 1)
portfoliosim['sharpeTot'] = [portfoliosimsharpeexp] + [0] * (len(portfoliosim) - 1)
portfolio['sharpeTot'] = portfolio['sharpeTot'].round(3)
portfoliosim['sharpeTot'] = portfoliosim['sharpeTot'].round(3)
# display (portfolio)
# display(portfoliosim)

***************************************************************************
MAXIMUM AND TARGET SHARPES
***************************************************************************

In [ ]:
# MAXIMUM SHARPE calculation
res_max, w_max = maxSharpeWeights(mu, cov, riskfree)
ressim_max, wsim_max = maxSharpeWeights(musim, covsim, riskfree)
print(f'Maximum Target Sharpe Routine =', res_max)
print(f'Maximum Target Sharpe Routine Simuation =', ressim_max)
# print("Max Sharpe:", portfolioPerformance(w_max, mu, cov, riskfree)[2], "weights:", w_max)
# print("Ret Max Sharpe:", w_max)

In [ ]:
#  add Maximum Sharpe ticker weights to portfolio dataframe
portfolio['sharpeMax'] = w_max.round(3) * 100
portfoliosim['sharpeMax'] = wsim_max.round(3) * 100
# display(portfolio)

In [ ]:
# add Maximum Sharpe value to portfolio dataframe in the first line
portfolio.at[0, 'sharpeMax'] = round(portfolioPerformance(w_max, mu, cov, riskfree)[2], 3)
portfoliosim.at[0, 'sharpeMax'] = round(portfolioPerformance(wsim_max, musim, covsim, riskfree)[2], 3)
# display(portfolio)

In [ ]:
# add Sharpe Maximum Return column and the value in the first line
portfolio['returnSharpeMax'] = [portfolioPerformance(w_max, mu, cov, riskfree)[0] *100] + [0] * (len(portfolio) - 1)
portfoliosim['returnSharpeMax'] = [portfolioPerformance(wsim_max, musim, covsim, riskfree)[0] *100] + [0] * (len(portfoliosim) - 1)
portfolio['returnSharpeMax'] = portfolio['returnSharpeMax'].round(1)
portfoliosim['returnSharpeMax'] = portfoliosim['returnSharpeMax'].round(1)
# display(portfolio)
# display(portfoliosim)

In [ ]:
# TARGET SHARPE calculation as a percentage of MAXIMUM TARGET (ShMaxE-W)
target = portfolio.at[0, 'sharpeMax'] * target_sharpe
targetsim = portfoliosim.at[0, 'sharpeMax'] * target_sharpesim
print(f'Target Portfolio =', target)
print(f'Target Portfolio Simulation =', targetsim)

# CORRECT call: pass limitsport (in fraction 0..1) as argument
res_tgt, w_tgt, info = targetSharpeWeights(mu, cov, target, riskfree, port_limits)
ressim_tgt, wsim_tgt, infosim = targetSharpeWeights(musim, covsim, targetsim, riskfree, portsim_limits)

# Debug/output (optional)
print(f'Target Portfolio =', res_tgt)
print(f'Target Portfolio Simulation =', res_tgt)
# print("Target result:", info)
# print("Weights:", w_tgt)
# print("Achieved Sharpe:", info.get('achieved_sharpe'))

In [ ]:
#  Add Target Sharpe ticker weights to portfolio dataframe
portfolio['sharpeTarget'] = (w_tgt * 100).round(3)
portfoliosim['sharpeTarget'] = (wsim_tgt * 100).round(3)
# Add Target (Achieved) Sharpe value to portfolio dataframe in the first line
portfolio.at[0, 'sharpeTarget'] = round(info['achieved_sharpe'], 3)
portfoliosim.at[0, 'sharpeTarget'] = round(info['achieved_sharpe'], 3)
# Add Target Maximum Return column and the value in the first line
portfolio['returnSharpeTarget'] = [portfolioPerformance(w_tgt, mu, cov, riskfree)[0] *100] + [0] * (len(portfolio) - 1)
portfoliosim['returnSharpeTarget'] = [portfolioPerformance(wsim_tgt, musim, covsim, riskfree)[0] *100] + [0] * (len(portfoliosim) - 1)
portfolio['returnSharpeTarget'] = portfolio['returnSharpeTarget'].round(1)
portfoliosim['returnSharpeTarget'] = portfoliosim['returnSharpeTarget'].round(1)
# display(portfolio)
# display(portfoliosim)

***************************************************************************
GOOGLE SHEETS HISTORIC SERIES RESULTS WRITINGS
***************************************************************************

In [ ]:
# open workbook and worksheets
wb = gc.open('portfolio')
wsport = wb.worksheet('portfolio')
wsportsim = wb.worksheet('portfoliosim')
wsstockbr = wb.worksheet('stockbrHist')
wsreitbr = wb.worksheet('reitbrHist')
wsstockus = wb.worksheet('stockusHist')
wsetfus = wb.worksheet('etfusHist')
wsreitus = wb.worksheet('reitusHist')

In [ ]:
# write data in the worksheets (Método definitivo sem avisos e sem quebras)
wb.values_update('portfolio!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [portfolio.columns.tolist()] + portfolio.values.tolist()})
wb.values_update('portfoliosim!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [portfoliosim.columns.tolist()] + portfoliosim.values.tolist()})
wb.values_update('stockbrHist!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [stockbr_factors.columns.tolist()] + stockbr_factors.values.tolist()})
wb.values_update('reitbrHist!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [reitbr_factors.columns.tolist()] + reitbr_factors.values.tolist()})
wb.values_update('stockusHist!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [stockus_factors.columns.tolist()] + stockus_factors.values.tolist()})
wb.values_update('etfusHist!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [etfus_factors.columns.tolist()] + etfus_factors.values.tolist()})
wb.values_update('reitusHist!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [reitus_factors.columns.tolist()] + reitus_factors.values.tolist()})

***************************************************************************
ASSETS FUNDAMENTALISTS INFOS READINGS AND GOOGLE SHEETS WRITINGS
***************************************************************************

In [ ]:
# use from invest.yahoofinance import yfStock
if fund_reading == 'Yes':
    # YAHOO FINANCE STOCK BR FUNDAMENTALISTS INFO READING
    stockbrFund = yfStock(stockbr_ticker)
    # display(stockbrFund)

In [ ]:
if fund_reading == 'Yes':
# write
    wb = gc.open('portfolio')
    wb.values_update('stockbrFund!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [stockbrFund.columns.tolist()] + stockbrFund.values.tolist()})

In [ ]:
# use from invest.reitbr import reitbrData, reitbrDataFund
if fund_reading == 'Yes':
    # FUNDS EXPLORER REAL STATE BR INFO READING
    reitbrFund = reitbrDataFund('https://www.fundsexplorer.com.br/ranking') # use this line to read directly from Funds Explorer site (scraping), it is the best option
    # reitbrFund = reitbrData('fundsexplorer.xlsx', reitbr_ticker) # excel file reading using VS CODE ONLY to Google Drive access, comment this line in case not using VS Code
    # reitbrFund = pd.read_excel('fundsexplorer.xlsx') # excel file reading using COLAB ONLY to Google Drive access, comment this line in case not using Colab
    # display(reitbrFund)

In [ ]:
if fund_reading == 'Yes':
    # list filter, exclude tickers not in real state list
    reitbr_ticker_adjusted = [nome.replace('.SA', '') for nome in reitbr_ticker]
    reitbrFund = reitbrFund[reitbrFund['Fundos'].isin(reitbr_ticker_adjusted)]
    reitbrFund = reitbrFund.reset_index(drop=True)
    # replace NaN values with None in reitbr_info before updating the sheet
    reitbrFund = reitbrFund.replace({np.nan: None})
    # display(reitbrFund)

In [ ]:
if fund_reading == 'Yes':
# write
    wb = gc.open('portfolio')
    wb.values_update('reitbrFund!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [reitbrFund.columns.tolist()] + reitbrFund.values.tolist()})

In [ ]:
# use from invest.yahoofinance import yfStock
if fund_reading == 'Yes':
    # YAHOO FINANCE STOCK US FUNDAMENTALISTS INFO READING
    stockusFund = yfStock(stockus_ticker)
    # display(stockusFund)

In [ ]:
if fund_reading == 'Yes':
# write
    wb = gc.open('portfolio')
    wb.values_update('stockusFund!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [stockusFund.columns.tolist()] + stockusFund.values.tolist()})

In [ ]:
# use from invest.yahoofinance import yfEtf
if fund_reading == 'Yes':
    # YAHOO FINANCE ETF US FUNDAMENTALISTS INFO READING
    etfusFund = yfEtf(etfus_ticker)
    # display(etftusFund)

In [ ]:
if fund_reading == 'Yes':
    # write
    wb = gc.open('portfolio')
    wb.values_update('etfusFund!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [etfusFund.columns.tolist()] + etfusFund.values.tolist()})

In [ ]:
# use from invest.yahoofinance import yfEtf
if fund_reading == 'Yes':
    # YAHOO FINANCE REIT US FUNDAMENTALISTS INFO READING
    reitusFund = yfEtf(reitus_ticker)
    # display(reitusFundFund)

In [ ]:
if fund_reading == 'Yes':
    # write
    wb = gc.open('portfolio')
    wb.values_update('reitusFund!A1', params={'valueInputOption': 'USER_ENTERED'}, body={'values': [reitusFund.columns.tolist()] + reitusFund.values.tolist()})

In [ ]:
print("END")

In [ ]:
# ⚙️ SETUP tips

# !pip install selenium webdriver-manager beautifulsoup4 -q

# Remove Chromium
# !apt-get remove -y chromium-browser chromium-chromedriver -qq 2>/dev/null

# oficial Chrome install
# !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# !apt-get install -y -qq ./google-chrome-stable_current_amd64.deb 2>/dev/null
# !rm google-chrome-stable_current_amd64.deb

# ChromeDriver install, compatilble with Chrome
# !pip install chromedriver-binary-auto -q

# !google-chrome --version